In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import torch
from dataset_class.job_post_dataset import JobPostingDataset
from sklearn.model_selection import train_test_split

### Splitting 

This section serves to split the dataset into Train, validation and test sets

In [ ]:
### loading clean text

df = pd.read_csv("./data/clean/fake_job_postings_nlp.csv")

df.head()



In [ ]:
# 1. Split 80% Train, 20% "Rest" (temp_data)
train_data, temp_data = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

# 2. Split that 20% into half (10% Val, 10% Test)
# FIX: Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

In [ ]:
X_train = train_data['full_text']
y_train = train_data['fraudulent']

X_val = val_data['full_text']
y_val = val_data['fraudulent']

X_test = test_data['full_text']
y_test = test_data['fraudulent']

### Tokenization

This sections serves to tokenize free text into a sequence of integers

In [ ]:
### Using Hugging Face's AutoTokenizer to tokenize the cleaned text data

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")

def longformer_encode(text, max_length=2048):
    return tokenizer.encode_plus(
        text,
        add_special_tokens=True,  # Add [CLS] and [SEP]
        max_length=max_length,
        padding='max_length',  # Pad to max_length
        truncation=True,  # Truncate if longer than max_length
        return_attention_mask=True,  # Return attention mask
        return_tensors='pt'  # Return PyTorch tensors
    )

In [ ]:
# Example : 

sample_text = df['full_text'].iloc[0]
encoded_input = longformer_encode(sample_text)

print(f"Tokens: {tokenizer.convert_ids_to_tokens(encoded_input['input_ids'][0])}")
print("Input IDs shape:", encoded_input['input_ids'].shape)

In [ ]:
# Create the dataset object
train_dataset = JobPostingDataset(
    texts=X_train.to_list(),
    labels=y_train.to_list(),
    tokenizer=tokenizer,
    max_len=2048
)

val_dataset = JobPostingDataset(
    texts=X_val.to_list(),
    labels=y_val.to_list(),
    tokenizer=tokenizer,
    max_len=2048
)


test_dataset = JobPostingDataset(
    texts=X_test.to_list(),
    labels=y_test.to_list(),
    tokenizer=tokenizer,
    max_len=2048
)

In [ ]:
## Test dataset

print(f"Dataset length: {len(train_dataset)}")
print(f"Dataset length: {len(val_dataset)}")
print(f"Dataset length: {len(test_dataset)}")


In [ ]:
# Access the first item
item = train_dataset[0] 

print(f"Input IDs shape: {item['input_ids'].shape}")         # torch.Size([2048])
print(f"Attention Mask shape: {item['attention_mask'].shape}") # torch.Size([2048])
print(f"Global Mask shape: {item['global_attention_mask'].shape}") # torch.Size([2048])
print(f"Label: {item['labels']}")                            # tensor(0) or tensor(1)


### Embeddings

This section serves to convert the token ids into a high-dimensional vector to capture the semantic and syntactic meaning of the tokens